# Task 4 — Open-Set Recognition (Colab)

CIFAR-10 known / CIFAR-100 near–far unknowns. Methods: **Vanilla**, **GCSC** (RandAugment), **PROSER**. Scores: MSP / MLS / Energy / Mahalanobis (+ PROSER placeholder).

**Runtime:** GPU (T4). Keep this tab open while training cells run.

Outputs land in `task4/results/` (checkpoints, curves, tables, figures, cache).


## 0) GPU check


In [1]:
import torch
print("cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Runtime → Change runtime type → T4 GPU, then re-run.")
print("device:", torch.cuda.get_device_name(0))


cuda: True
device: Tesla T4


## 1) Mount Drive and enter the repo


In [2]:
from pathlib import Path
from google.colab import drive
import os

drive.mount("/content/drive")

# Adjust if your Drive path differs:
REPO = Path("/content/drive/MyDrive/MS AI/Semester_3/ATML/PAs/ATML-PA1")
assert REPO.exists(), f"Repo not found: {REPO}"
os.chdir(REPO)
print("cwd:", Path.cwd())
print("task4:", (REPO / "task4").exists())


Mounted at /content/drive
cwd: /content/drive/MyDrive/MS AI/Semester_3/ATML/PAs/ATML-PA1
task4: True


## 2) Install dependencies


In [3]:
%pip install -q -r requirements.txt


## 3) Helpers

Skip a training stage only when both `*_best.pt` and `*_train_summary.json` exist.


In [4]:
from pathlib import Path

RESULTS = Path("task4/results")

def stage_done(name: str) -> bool:
    pt = RESULTS / "checkpoints" / f"{name}_best.pt"
    summary = RESULTS / "tables" / f"{name}_train_summary.json"
    return pt.exists() and summary.exists()

print("vanilla done?", stage_done("vanilla"))
print("gcsc done?", stage_done("gcsc"))
print("proser done?", stage_done("proser"))


vanilla done? False
gcsc done? False
proser done? False


## 4) Splits (seed 6304)

Creates `task4/results/splits/cifar10_seed6304.json` if missing.


In [5]:
!python -m task4.scripts.run_task4 --stages splits


>> splits ready → task4/results/splits/cifar10_seed6304.json
seed=6304 train=45000 val=5000 test=10000


## 5) Train Vanilla (100 epochs)

Checkpoint selection uses **val Acc only**. Re-running skips if already complete.


In [6]:
if stage_done("vanilla"):
    print("SKIP train_vanilla — checkpoint + summary already present")
else:
    !python -m task4.scripts.run_task4 --stages train_vanilla

import torch, json
from pathlib import Path
ck = Path("task4/results/checkpoints/vanilla_best.pt")
meta = torch.load(ck, map_location="cpu", weights_only=False).get("meta", {})
summary = json.loads(Path("task4/results/tables/vanilla_train_summary.json").read_text())
print("Vanilla meta:", meta)
print("Vanilla summary best_val_acc:", summary.get("best_val_acc"), "test_csa:", summary.get("test_csa"))


>> /usr/bin/python3 -m task4.train --method-config task4/configs/vanilla.yaml
device=cuda method=vanilla
[vanilla] best checkpoint kept at epoch 98 val_acc=0.9544
Vanilla meta: {'method': 'vanilla', 'epoch': 98, 'val_acc': 0.9544, 'randaugment': False}
Vanilla summary best_val_acc: 0.9544 test_csa: 0.9472


## 6) Train GCSC (Vanilla + RandAugment, 100 epochs)


In [7]:
if stage_done("gcsc"):
    print("SKIP train_gcsc — already complete")
else:
    !python -m task4.scripts.run_task4 --stages train_gcsc


>> /usr/bin/python3 -m task4.train --method-config task4/configs/gcsc.yaml
device=cuda method=gcsc
[gcsc] epoch 1/100 loss=2.1275 val_acc=0.3270
[gcsc] epoch 2/100 loss=1.6527 val_acc=0.4770
[gcsc] epoch 3/100 loss=1.3746 val_acc=0.5498
[gcsc] epoch 4/100 loss=1.1802 val_acc=0.6448
[gcsc] epoch 5/100 loss=1.0259 val_acc=0.6380
[gcsc] epoch 6/100 loss=0.8938 val_acc=0.7440
[gcsc] epoch 7/100 loss=0.8030 val_acc=0.7378
[gcsc] epoch 8/100 loss=0.7434 val_acc=0.7750
[gcsc] epoch 9/100 loss=0.7019 val_acc=0.7900
[gcsc] epoch 10/100 loss=0.6671 val_acc=0.8144
[gcsc] epoch 11/100 loss=0.6530 val_acc=0.6908
[gcsc] epoch 12/100 loss=0.6289 val_acc=0.7958
[gcsc] epoch 13/100 loss=0.6178 val_acc=0.8142
[gcsc] epoch 14/100 loss=0.5928 val_acc=0.7650
[gcsc] epoch 15/100 loss=0.5766 val_acc=0.8180
[gcsc] epoch 16/100 loss=0.5670 val_acc=0.7972
[gcsc] epoch 17/100 loss=0.5546 val_acc=0.7998
[gcsc] epoch 18/100 loss=0.5515 val_acc=0.8434
[gcsc] epoch 19/100 loss=0.5387 val_acc=0.8438
[gcsc] epoch 20/1

## 7) Train PROSER (50 epochs, init from Vanilla)

Requires `vanilla_best.pt`.


In [8]:
from pathlib import Path
if stage_done("proser"):
    print("SKIP train_proser — already complete")
else:
    assert Path("task4/results/checkpoints/vanilla_best.pt").exists()
    !python -m task4.scripts.run_task4 --stages train_proser


>> /usr/bin/python3 -m task4.train --method-config task4/configs/proser.yaml
device=cuda method=proser
[proser] epoch 1/50 loss=0.3317 val_acc=0.9498
[proser] epoch 2/50 loss=0.3022 val_acc=0.9474
[proser] epoch 3/50 loss=0.2863 val_acc=0.9500
[proser] epoch 4/50 loss=0.2675 val_acc=0.9484
[proser] epoch 5/50 loss=0.2330 val_acc=0.9482
[proser] epoch 6/50 loss=0.1657 val_acc=0.9452
[proser] epoch 7/50 loss=0.1117 val_acc=0.9470
[proser] epoch 8/50 loss=0.1004 val_acc=0.9446
[proser] epoch 9/50 loss=0.0833 val_acc=0.9450
[proser] epoch 10/50 loss=0.0732 val_acc=0.9394
[proser] epoch 11/50 loss=0.0795 val_acc=0.9428
[proser] epoch 12/50 loss=0.0696 val_acc=0.9402
[proser] epoch 13/50 loss=0.0569 val_acc=0.9386
[proser] epoch 14/50 loss=0.0708 val_acc=0.9422
[proser] epoch 15/50 loss=0.0818 val_acc=0.9410
[proser] epoch 16/50 loss=0.0539 val_acc=0.9368
[proser] epoch 17/50 loss=0.0517 val_acc=0.9362
[proser] epoch 18/50 loss=0.0414 val_acc=0.9386
[proser] epoch 19/50 loss=0.0520 val_acc=0

## 8) Extract logits / features (known + near/far unknowns)


In [9]:
!python -m task4.scripts.run_task4 --stages extract_all


>> /usr/bin/python3 -m task4.extract_outputs --method vanilla
[vanilla] extract known/train_eval
[vanilla] extract known/val
[vanilla] extract known/test
[vanilla] extract unknown/near
[vanilla] extract unknown/far
[vanilla] cache → task4/results/cache/vanilla
>> /usr/bin/python3 -m task4.extract_outputs --method gcsc
[gcsc] extract known/train_eval
[gcsc] extract known/val
[gcsc] extract known/test
[gcsc] extract unknown/near
[gcsc] extract unknown/far
[gcsc] cache → task4/results/cache/gcsc
>> /usr/bin/python3 -m task4.extract_outputs --method proser
[proser] extract known/train_eval
[proser] extract known/val
[proser] extract known/test
[proser] extract unknown/near
[proser] extract unknown/far
[proser] cache → task4/results/cache/proser


## 9) OSR evaluation (scores, model table, figure, failures)


In [10]:
!python -m task4.scripts.run_task4 --stages eval

from pathlib import Path
import json
print("--- model comparison (MLS) ---")
rows = json.loads(Path("task4/results/tables/task4_model_comparison.json").read_text())["rows"]
for r in rows:
    if r["score"] == "mls":
        print(
            f"{r['model']:8s} CSA={r['csa']:.4f}  "
            f"AUROC_near={r['auroc_near']:.4f}  "
            f"AUROC_far={r['auroc_far']:.4f}  "
            f"AUROC_all={r['auroc_all']:.4f}"
        )
print("figure:", Path("task4/results/figures/score_distributions.png").exists())


>> /usr/bin/python3 -m task4.evaluate_osr --stages scores,models,figures,failures
Evaluating vanilla / msp
Evaluating vanilla / mls
Evaluating vanilla / energy
Evaluating vanilla / mahalanobis
Wrote task4/results/tables/task4_score_comparison.json
Evaluating vanilla / mls
Evaluating gcsc / mls
Evaluating proser / mls
Evaluating proser / proser_placeholder
Wrote task4/results/tables/task4_model_comparison.json
Wrote task4/results/figures/score_distributions.png
Wrote task4/results/tables/task4_failure_analysis.json (6 examples)
--- model comparison (MLS) ---
vanilla  CSA=0.9472  AUROC_near=0.7982  AUROC_far=0.8870  AUROC_all=0.8426
gcsc     CSA=0.9512  AUROC_near=0.8152  AUROC_far=0.9053  AUROC_all=0.8602
proser   CSA=0.9442  AUROC_near=0.8157  AUROC_far=0.8918  AUROC_all=0.8537
figure: True


## Done

Artifacts under `task4/results/`:

| Folder | Contents |
|--------|----------|
| `checkpoints/` | `vanilla_best.pt`, `gcsc_best.pt`, `proser_best.pt` |
| `curves/` | per-method training histories |
| `tables/` | train summaries + score/model/failure JSONs |
| `figures/` | `score_distributions.png` |
| `cache/` | logits/features for eval |
| `splits/` | `cifar10_seed6304.json` |

**Note:** Vanilla training hit a Colab GPU quota limit near epoch 98; the best checkpoint (`val_acc=0.9544`) was kept and used for extract/eval (`test_csa=0.9472`). GCSC and PROSER completed full schedules.
